In [11]:
import os, json, glob, time, gc
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from scipy.optimize import linear_sum_assignment

# ---- device: probe Conv3d on GPU 0 specifically (P100/sm_60 passes is_available but fails on conv) ----
DEVICE = 'cpu'
N_GPUS = 0

if torch.cuda.is_available():
    try:
        _p = nn.Conv3d(1, 1, 3).cuda()
        _ = _p(torch.zeros(1, 1, 8, 8, 8, device='cuda')).cpu()
        del _p
        DEVICE = 'cuda'
        N_GPUS = torch.cuda.device_count()
    except Exception as e:
        print(f'GPU unusable (likely P100/sm_60): {str(e)[:80]} -> CPU')

print(f'Device: {DEVICE} | GPUs available: {N_GPUS} | PyTorch: {torch.__version__}')
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB VRAM')

# ---- paths ----
CAND = [
    '/kaggle/input/competitions/biohub-cell-tracking-during-development',
    '/kaggle/input/biohub-cell-tracking-during-development',
]
ROOT = next((p for p in CAND if Path(p, 'train').exists()), None)
if ROOT is None:
    raise FileNotFoundError(f'Data not found. Checked: {CAND}')
TRAIN_DIR = Path(ROOT) / 'train'
TEST_DIR  = Path(ROOT) / 'test'
OUT_DIR   = Path('/kaggle/working')
CKPT_PATH = OUT_DIR / 'unet3d.pt'
print(f'Data root: {ROOT}')

# ---- training knobs ----
POOL         = 4
BASE         = 24
EPOCHS       = 15
FPM          = 16
# double batch size when 2 GPUs: each GPU still sees 16 samples,
# total throughput doubles, same memory per device
BATCH        = 16 * max(N_GPUS, 1)
LR           = 1e-3 * max(N_GPUS, 1)   # linear LR scaling with batch size
SEED         = 42

# ---- positive-unlabelled loss weights ----
GAUSS_SIGMA  = 1.0
POS_THRESH   = 0.05
BG_QUANTILE  = 0.40
W_POS        = 12.0
W_BG         = 1.0
W_IGN        = 0.05

TEST4 = ['6bba_05b6850b', '6bba_05db0fb1', '44b6_0113de3b', '44b6_0b24845f']

RNG = np.random.default_rng(SEED)
torch.manual_seed(SEED)

print(f'Effective batch size: {BATCH} ({BATCH // max(N_GPUS,1)} per GPU x {max(N_GPUS,1)} GPUs)')
print(f'Learning rate: {LR:.1e}')

Device: cuda | GPUs available: 2 | PyTorch: 2.10.0+cu128
  GPU 0: Tesla T4 | 15.6 GB VRAM
  GPU 1: Tesla T4 | 15.6 GB VRAM
Data root: /kaggle/input/competitions/biohub-cell-tracking-during-development
Effective batch size: 32 (16 per GPU x 2 GPUs)
Learning rate: 2.0e-03


In [12]:
def _block(ci, co):
    return nn.Sequential(
        nn.Conv3d(ci, co, 3, padding=1),
        nn.BatchNorm3d(co),
        nn.ReLU(inplace=True),
        nn.Conv3d(co, co, 3, padding=1),
        nn.BatchNorm3d(co),
        nn.ReLU(inplace=True),
    )

class UNet3D(nn.Module):
    def __init__(self, base=BASE):
        super().__init__()
        b = base
        self.e1 = _block(1,   b);    self.e2 = _block(b,   b*2)
        self.e3 = _block(b*2, b*4);  self.pool = nn.MaxPool3d(2)
        self.bn = _block(b*4, b*8)
        self.u3 = nn.ConvTranspose3d(b*8, b*4, 2, stride=2)
        self.d3 = _block(b*8, b*4)
        self.u2 = nn.ConvTranspose3d(b*4, b*2, 2, stride=2)
        self.d2 = _block(b*4, b*2)
        self.u1 = nn.ConvTranspose3d(b*2, b,   2, stride=2)
        self.d1 = _block(b*2, b)
        self.out = nn.Conv3d(b, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b  = self.bn(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

# build on CPU first, then move — DataParallel requires the model
# to already be on cuda:0 before wrapping
model = UNet3D(base=BASE).to(DEVICE)

if N_GPUS > 1:
    model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))
    print(f'DataParallel enabled across GPUs: {list(range(N_GPUS))}')
else:
    print(f'Single device: {DEVICE}')

n_params = sum(p.numel() for p in model.parameters())
print(f'UNet3D params: {n_params:,}')

# sanity forward pass
with torch.no_grad():
    _t = torch.zeros(BATCH, 1, 64, 64, 64, device=DEVICE)
    _o = model(_t)
    print(f'Forward OK: {tuple(_t.shape)} -> {tuple(_o.shape)}')
    del _t, _o

DataParallel enabled across GPUs: [0, 1]
UNet3D params: 3,152,425
Forward OK: (32, 1, 64, 64, 64) -> (32, 1, 64, 64, 64)


In [4]:
_ZC = {}  # zarr array cache — avoids reopening per frame

def read_meta(zp):
    with open(Path(zp)/'0'/'zarr.json') as f:
        m = json.load(f)
    return dict(shape=tuple(m['shape']), dtype=np.dtype(m['data_type']))

def load_vol(zp, t, meta=None):
    """Load one timepoint (Z,Y,X). Uses zarr with blosc2 fallback."""
    try:
        import zarr
        k = str(zp)
        if k not in _ZC:
            _ZC[k] = zarr.open(k, mode='r')['0']
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None:
            meta = read_meta(zp)
        buf = blosc2.decompress(open(Path(zp)/'0'/'c'/str(t)/'0'/'0'/'0', 'rb').read())
        return np.frombuffer(buf, dtype=meta['dtype']).reshape(meta['shape'][1:])

def pool_xy(vol, f=POOL):
    """Block-mean XY pool -> ~isotropic voxels. No copy if already float32."""
    Z, Y, X = vol.shape
    Y2, X2 = (Y//f)*f, (X//f)*f
    v = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return v.reshape(Z, Y2//f, f, X2//f, f).mean(axis=(2, 4))

def normalize(p):
    """Robust [p50, p99.5] -> [0,1] normalization, clip to [-0.5, 6]."""
    lo = float(np.percentile(p, 50.0))
    hi = float(np.percentile(p, 99.5))
    return np.clip((p - lo) / (hi - lo + 1e-6), -0.5, 6.0).astype(np.float32)

def read_geff_nodes(geff_path):
    """Read GT node coords -> (N, 4) array of (t, z, y, x) in voxel units."""
    try:
        import zarr
        root = zarr.open(str(geff_path), mode='r')
        cols = [np.asarray(root[f'nodes/props/{c}/values']) for c in ('t','z','y','x')]
    except Exception:
        import zstandard
        def _dec(path):
            path = Path(path)
            m = json.load(open(path/'zarr.json'))
            dt = np.dtype(m['data_type']); n = int(m['shape'][0])
            buf = zstandard.ZstdDecompressor().decompress(open(path/'c'/'0','rb').read())
            return np.frombuffer(buf, dtype=dt)[:n]
        cols = [_dec(f'{geff_path}/nodes/props/{c}/values') for c in ('t','z','y','x')]
    return np.stack(cols, axis=1).astype(np.float64)

In [5]:
def all_movies():
    zarr_names = {Path(p).name[:-5] for p in glob.glob(f'{TRAIN_DIR}/*.zarr')}
    geff_names = {Path(p).name[:-5] for p in glob.glob(f'{TRAIN_DIR}/*.geff')}
    return sorted(zarr_names & geff_names)

def make_splits(n_val_per_embryo=8, seed=SEED):
    import collections, random
    pool = [m for m in all_movies() if m not in TEST4]
    by_embryo = collections.defaultdict(list)
    for m in pool:
        by_embryo[m.split('_')[0]].append(m)
    rng = random.Random(seed)
    train_set, val_set = [], []
    for emb in sorted(by_embryo):
        ms = sorted(by_embryo[emb])
        rng.shuffle(ms)
        val_set.extend(ms[:n_val_per_embryo])
        train_set.extend(ms[n_val_per_embryo:])
    return train_set, val_set

train_movies, val_movies = make_splits(n_val_per_embryo=8)
print(f'Train: {len(train_movies)} movies | Val: {len(val_movies)} movies | Held-out test4: {len(TEST4)}')

Train: 179 movies | Val: 16 movies | Held-out test4: 4


In [6]:
# Cache all training frames in RAM upfront — this is the key CPU bottleneck fix.
# Streaming from disk per batch stalls the GPU constantly; caching once lets
# the GPU run at full utilization for the rest of training.
# 16 frames/movie * ~171 movies * ~64^3 * 4 bytes ≈ 1.7GB — well within 30GB RAM limit.

def load_movie_frames(ds, fpm):
    """Return list of (input_array [1,D,H,W] f32, gt_pts Nx3) for fpm evenly-sampled GT frames."""
    zp = str(TRAIN_DIR / f'{ds}.zarr')
    gp = str(TRAIN_DIR / f'{ds}.geff')
    nodes = read_geff_nodes(gp)
    meta  = read_meta(zp)
    node_t = nodes[:, 0].round().astype(int)
    ts = sorted(set(node_t.tolist()) & set(range(min(meta['shape'][0], 100))))
    if not ts:
        return []
    idx = np.linspace(0, len(ts)-1, min(fpm, len(ts))).round().astype(int)
    out = []
    for t in sorted(set(ts[i] for i in idx)):
        vol = load_vol(zp, t, meta)
        x   = normalize(pool_xy(vol))  # (D, H/4, W/4) f32
        sel = nodes[node_t == t]
        # convert GT coords to pooled-grid space
        pts = np.stack([sel[:,1], sel[:,2]/POOL, sel[:,3]/POOL], axis=1).astype(np.float32)
        out.append((x[None], pts))     # x[None] -> (1,D,H,W)
    return out

t0 = time.time()
train_cache = []
for i, ds in enumerate(train_movies):
    train_cache += load_movie_frames(ds, FPM)
    if (i+1) % 20 == 0:
        print(f'  cached {i+1}/{len(train_movies)} movies, {len(train_cache)} frames ({time.time()-t0:.0f}s)')
print(f'Train cache: {len(train_cache)} frames, {time.time()-t0:.0f}s total')

val_cache = []
for ds in val_movies:
    val_cache += load_movie_frames(ds, fpm=4)
print(f'Val cache: {len(val_cache)} frames')

assert len(train_cache) > 0, 'No training frames — check data root and GT reader'

# estimate RAM used
import sys
total_bytes = sum(f[0].nbytes for f in train_cache)
print(f'Train cache RAM: ~{total_bytes / 1e9:.2f} GB')

  cached 20/179 movies, 320 frames (30s)
  cached 40/179 movies, 640 frames (59s)
  cached 60/179 movies, 960 frames (87s)
  cached 80/179 movies, 1280 frames (115s)
  cached 100/179 movies, 1600 frames (142s)
  cached 120/179 movies, 1920 frames (170s)
  cached 140/179 movies, 2240 frames (199s)
  cached 160/179 movies, 2560 frames (226s)
Train cache: 2864 frames, 253s total
Val cache: 64 frames
Train cache RAM: ~3.00 GB


In [7]:
def stamp_heatmap(shape, pts, sigma=GAUSS_SIGMA):
    """Gaussian-stamp GT centroids into a heatmap. max-composite so blobs don't sum."""
    hm = np.zeros(shape, np.float32)
    r  = int(np.ceil(3 * sigma))
    zz, yy, xx = np.mgrid[-r:r+1, -r:r+1, -r:r+1]
    g  = np.exp(-(zz**2 + yy**2 + xx**2) / (2 * sigma**2)).astype(np.float32)
    D, H, W = shape
    for z, y, x in pts:
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z-r), min(D, z+r+1)
        y0, y1 = max(0, y-r), min(H, y+r+1)
        x0, x1 = max(0, x-r), min(W, x+r+1)
        gz = z0 - (z-r); gy = y0 - (y-r); gx = x0 - (x-r)
        sub = g[gz:gz+(z1-z0), gy:gy+(y1-y0), gx:gx+(x1-x0)]
        np.maximum(hm[z0:z1, y0:y1, x0:x1], sub, out=hm[z0:z1, y0:y1, x0:x1])
    return hm

def build_sample(x, pts, augment=True):
    """
    Returns (x, target, weight) arrays ready for the loss.
    Weight encodes positive-unlabelled scheme:
      - GT centroid voxels (heatmap > POS_THRESH): W_POS = 12
      - Confidently dark voxels (< BG_QUANTILE):  W_BG  = 1
      - Everything else (bright but unlabelled):  W_IGN = 0.05
    """
    x = x.copy()
    tgt = stamp_heatmap(x.shape[1:], pts)
    if augment:
        # flip Y and X axes independently (axes 2,3 of [1,D,H,W])
        for ax in (2, 3):
            if RNG.random() < 0.5:
                x   = np.flip(x,   ax).copy()
                tgt = np.flip(tgt, ax-1).copy()
    w = np.full(tgt.shape, W_IGN, np.float32)
    w[x[0] < np.quantile(x[0], BG_QUANTILE)] = W_BG   # dark -> confirmed background
    w[tgt > POS_THRESH] = W_POS                        # GT centre -> strong positive
    return x, tgt, w

In [8]:
from skimage.feature import peak_local_max

def compute_val_recall(model, val_cache, thresh=0.3, tol=2.0):
    """
    Fraction of GT centroids within `tol` voxels of a predicted peak.
    tol=2.0 in downsampled-grid voxels ≈ 3.3um in Z, ≈ 0.8um in XY -- tight.
    """
    model.eval()
    hit = tot = 0
    with torch.no_grad():
        for x, pts in val_cache:
            xt = torch.from_numpy(x)[None].to(DEVICE, non_blocking=True)
            hm = torch.sigmoid(model(xt))[0, 0].float().cpu().numpy()
            pk = peak_local_max(hm, min_distance=1, threshold_abs=thresh,
                                exclude_border=False).astype(np.float32)
            for p in pts:
                tot += 1
                if len(pk) and np.min(np.linalg.norm(pk - p, axis=1)) <= tol:
                    hit += 1
    model.train()
    return hit / max(tot, 1)

In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
bce_loss  = nn.BCEWithLogitsLoss(reduction='none')

# pinned memory tensors for zero-copy CPU->GPU transfers
# pre-allocate once, reuse every batch — eliminates per-batch allocation overhead
if DEVICE == 'cuda':
    _xb_pin  = torch.zeros(BATCH, 1, 64, 64, 64, pin_memory=True)
    _yb_pin  = torch.zeros(BATCH, 1, 64, 64, 64, pin_memory=True)
    _wb_pin  = torch.zeros(BATCH, 1, 64, 64, 64, pin_memory=True)

best_recall = -1.0
best_state  = None
n = len(train_cache)
t_start = time.time()

for epoch in range(EPOCHS):
    model.train()
    perm      = RNG.permutation(n)
    epoch_loss = 0.0
    n_steps    = 0

    for i in range(0, n, BATCH):
        batch_idx = perm[i:i+BATCH]
        samples   = [build_sample(*train_cache[j], augment=True) for j in batch_idx]
        bs        = len(samples)

        xb_np = np.stack([s[0] for s in samples])   # (B,1,D,H,W)
        yb_np = np.stack([s[1] for s in samples])   # (B,D,H,W)
        wb_np = np.stack([s[2] for s in samples])   # (B,D,H,W)

        if DEVICE == 'cuda':
            # non_blocking=True + pinned memory = async CPU->GPU copy,
            # overlaps with next batch's CPU-side augmentation
            xb = torch.from_numpy(xb_np).to(DEVICE, non_blocking=True)
            yb = torch.from_numpy(yb_np[:, None]).to(DEVICE, non_blocking=True)
            wb = torch.from_numpy(wb_np[:, None]).to(DEVICE, non_blocking=True)
        else:
            xb = torch.from_numpy(xb_np)
            yb = torch.from_numpy(yb_np[:, None])
            wb = torch.from_numpy(wb_np[:, None])

        optimizer.zero_grad(set_to_none=True)   # set_to_none avoids zeroing memory, slightly faster
        logits = model(xb)
        loss   = (bce_loss(logits, yb) * wb).sum() / wb.sum()
        loss.backward()
        optimizer.step()

        epoch_loss += float(loss.detach()) * bs
        n_steps    += 1

    scheduler.step()

    avg_loss = epoch_loss / max(n_steps * BATCH, 1)
    recall   = compute_val_recall(model, val_cache)
    elapsed  = (time.time() - t_start) / 60

    if recall > best_recall:
        best_recall = recall
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        tag = ' <- best'
    else:
        tag = ''

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | loss={avg_loss:.4f} | '
          f'val_recall={recall:.3f} | lr={optimizer.param_groups[0]["lr"]:.1e} | '
          f'{elapsed:.1f}min{tag}')

    # safety: stop if over 55 minutes to leave time for saving + inference
    if elapsed > 55:
        print(f'Stopping early at {elapsed:.1f}min to stay within 1-hour budget')
        break

print(f'\nTraining complete. Best val_recall: {best_recall:.4f}')

Epoch  1/15 | loss=0.1727 | val_recall=0.159 | lr=2.0e-03 | 3.2min <- best
Epoch  2/15 | loss=0.0517 | val_recall=0.012 | lr=1.9e-03 | 6.4min
Epoch  3/15 | loss=0.0347 | val_recall=0.126 | lr=1.8e-03 | 9.5min


KeyboardInterrupt: 

In [14]:
torch.save({
    'state_dict': best_state if best_state is not None else model.state_dict(),
    'base':       BASE,
    'pool':       POOL,
    'norm':       'p50-p99.5',
    'val_recall': best_recall,
    'n_train_movies': len(train_movies),
    'epochs_trained': epoch + 1,
    'config': {
        'gauss_sigma': GAUSS_SIGMA,
        'pos_thresh':  POS_THRESH,
        'bg_quantile': BG_QUANTILE,
        'w_pos': W_POS, 'w_bg': W_BG, 'w_ign': W_IGN,
    }
}, CKPT_PATH)

print(f'Saved: {CKPT_PATH}')
print(f'Best val_recall: {best_recall:.4f}')

# sanity reload
ck = torch.load(CKPT_PATH, map_location='cpu')
m2 = UNet3D(base=ck['base'])
m2.load_state_dict(ck['state_dict'])
m2.eval()
with torch.no_grad():
    y = m2(torch.zeros(1, 1, 64, 64, 64))
assert tuple(y.shape) == (1, 1, 64, 64, 64)
print(f'Reload + forward pass OK: {tuple(y.shape)}')

Saved: /kaggle/working/unet3d.pt
Best val_recall: 0.1594


RuntimeError: Error(s) in loading state_dict for UNet3D:
	Missing key(s) in state_dict: "e1.0.weight", "e1.0.bias", "e1.1.weight", "e1.1.bias", "e1.1.running_mean", "e1.1.running_var", "e1.3.weight", "e1.3.bias", "e1.4.weight", "e1.4.bias", "e1.4.running_mean", "e1.4.running_var", "e2.0.weight", "e2.0.bias", "e2.1.weight", "e2.1.bias", "e2.1.running_mean", "e2.1.running_var", "e2.3.weight", "e2.3.bias", "e2.4.weight", "e2.4.bias", "e2.4.running_mean", "e2.4.running_var", "e3.0.weight", "e3.0.bias", "e3.1.weight", "e3.1.bias", "e3.1.running_mean", "e3.1.running_var", "e3.3.weight", "e3.3.bias", "e3.4.weight", "e3.4.bias", "e3.4.running_mean", "e3.4.running_var", "bn.0.weight", "bn.0.bias", "bn.1.weight", "bn.1.bias", "bn.1.running_mean", "bn.1.running_var", "bn.3.weight", "bn.3.bias", "bn.4.weight", "bn.4.bias", "bn.4.running_mean", "bn.4.running_var", "u3.weight", "u3.bias", "d3.0.weight", "d3.0.bias", "d3.1.weight", "d3.1.bias", "d3.1.running_mean", "d3.1.running_var", "d3.3.weight", 

In [15]:
# inside the epoch loop, replace this line:
# best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# with this:
best_state = {k: v.cpu().clone() for k, v in unwrap_state_dict(model).items()}

NameError: name 'unwrap_state_dict' is not defined

In [16]:
torch.save({

    "epoch": epoch,

    "model_state_dict": model.state_dict(),

    "optimizer_state_dict": optimizer.state_dict(),

    "scheduler_state_dict": scheduler.state_dict(),

    "best_loss": best_loss,

    "train_loss": train_loss,

    "val_loss": val_loss,

}, "checkpoint.pth")

NameError: name 'best_loss' is not defined